
# Challenge #8: Aggregate Consumer Purchases

### Aggregate Consumer Purchases:

>For this week’s exercise we will look at customer purchase behavior to decide if we should offer a “Meal Deal” that would add a side and drink to a purchase of pizza or a burger. The incoming data is larger than usual for these exercises so I have packaged the workflow as an Alteryx Package.

### This week’s Objective:

>In order to decide if we should start including a new "Meal Deal" on our menu we want to study the potential impact on recent transactions. **Please identify the number and percentage of orders since July 1, 2013 which include the following categories of food: Pizza OR Burger along with a Side and Drink.**

### Summary of Data:

>Point of Sale data includes the ticket level information, and the lookup table categorizes items into higher level food categories.

#### Hint:

>Don't forget to join to the lookup table and filter by date. As always we look forward to your feedback and suggestions!

In [101]:
import pandas as pd
import datetime

# import and preview output

output = pd.read_csv('output.csv', index_col = False)
df_output = pd.DataFrame(output)
df_output.head(25)

,Potential Meal Deal,Total,%
0,10964,15497,70.749177


In [102]:
# import and preview input

input = pd.read_csv('input1.csv', index_col = False)
df = pd.DataFrame(input)
df

,TicketID,Date,MemberID,Desc,Price
0,100004,2013-01-07,NaN,Mozzarella Sticks,7.0
1,100004,2013-01-07,NaN,Jalapeno Poppers,7.0
2,100004,2013-01-07,NaN,Onion Rings,7.0
3,100004,2013-01-07,NaN,Onion Rings,7.0
4,100004,2013-01-07,NaN,Supreme - Small,9.0
...,...,...,...,...,...
269370,331462,2013-12-28,NaN,The Works - Medium,14.0
269371,331462,2013-12-28,NaN,Soda - Small,2.0
269372,331462,2013-12-28,NaN,Soda - Small,2.0
269373,331462,2013-12-28,NaN,Soda - Large,3.0


In [103]:
# import and preview input

input2 = pd.read_csv('input2.csv', index_col = False)
df2 = pd.DataFrame(input2)
df2.head(10)

,Desc,Type
0,Bacon Burger,Burger
1,Barbecue Chicken - Large,Pizza
2,Barbecue Chicken - Medium,Pizza
3,Barbecue Chicken - Small,Pizza
4,Buffalo Chicken - Large,Pizza
5,Buffalo Chicken - Medium,Pizza
6,Buffalo Chicken - Small,Pizza
7,Caprese Salad,Salad
8,Cheese Bread,Side
9,Cheese Burger,Burger


In [104]:
df.sort_values(by='Date')['Date'].unique()

array(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
       '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
       '2013-01-09', '2013-01-10', '2013-01-11', '2013-01-12',
       '2013-01-13', '2013-01-14', '2013-01-15', '2013-01-16',
       '2013-01-17', '2013-01-18', '2013-01-19', '2013-01-20',
       '2013-01-21', '2013-01-22', '2013-01-23', '2013-01-24',
       '2013-01-25', '2013-01-26', '2013-01-27', '2013-01-28',
       '2013-01-29', '2013-01-30', '2013-01-31', '2013-02-01',
       '2013-02-02', '2013-02-03', '2013-02-04', '2013-02-05',
       '2013-02-06', '2013-02-07', '2013-02-08', '2013-02-09',
       '2013-02-10', '2013-02-11', '2013-02-12', '2013-02-13',
       '2013-02-14', '2013-02-15', '2013-02-16', '2013-02-17',
       '2013-02-18', '2013-02-19', '2013-02-20', '2013-02-21',
       '2013-02-22', '2013-02-23', '2013-02-24', '2013-02-25',
       '2013-02-26', '2013-02-27', '2013-02-28', '2013-03-01',
       '2013-03-02', '2013-03-03', '2013-03-04', '2013-

In [105]:
df['Date'] = pd.to_datetime(df['Date'], format="%Y-%m-%d")
df = df[df['Date'] >= datetime.datetime(2013, 6, 1)]
df

,TicketID,Date,MemberID,Desc,Price
5528,101964,2013-06-03,NaN,Mozzarella Sticks,7.0
5529,101964,2013-06-03,NaN,Stromboli,7.5
5530,101964,2013-06-03,NaN,House Salad - Half,3.0
5531,101964,2013-06-03,NaN,House Salad - Whole,6.0
5532,101964,2013-06-03,NaN,Garden Fresh - Medium,14.0
...,...,...,...,...,...
269370,331462,2013-12-28,NaN,The Works - Medium,14.0
269371,331462,2013-12-28,NaN,Soda - Small,2.0
269372,331462,2013-12-28,NaN,Soda - Small,2.0
269373,331462,2013-12-28,NaN,Soda - Large,3.0


In [106]:
df.sort_values(by='Date')['Date'].unique()

<DatetimeArray>
['2013-06-01 00:00:00', '2013-06-02 00:00:00', '2013-06-03 00:00:00',
 '2013-06-04 00:00:00', '2013-06-05 00:00:00', '2013-06-06 00:00:00',
 '2013-06-07 00:00:00', '2013-06-08 00:00:00', '2013-06-09 00:00:00',
 '2013-06-10 00:00:00',
 ...
 '2013-12-22 00:00:00', '2013-12-23 00:00:00', '2013-12-24 00:00:00',
 '2013-12-25 00:00:00', '2013-12-26 00:00:00', '2013-12-27 00:00:00',
 '2013-12-28 00:00:00', '2013-12-29 00:00:00', '2013-12-30 00:00:00',
 '2013-12-31 00:00:00']
Length: 214, dtype: datetime64[ns]

- join data together
- group by order, aggregate item types
- filter to records with [[Pizza | Burger] & Side & Drink]
- count, get percent of total

In [107]:
df_merge = df.merge(df2, on='Desc')
df_merge.head(12)

,TicketID,Date,MemberID,Desc,Price,Type
0,101964,2013-06-03,NaN,Mozzarella Sticks,7.0,Side
1,101964,2013-06-03,NaN,Stromboli,7.5,Pizza
2,101964,2013-06-03,NaN,House Salad - Half,3.0,Salad
3,101964,2013-06-03,NaN,House Salad - Whole,6.0,Salad
4,101964,2013-06-03,NaN,Garden Fresh - Medium,14.0,Salad
5,101964,2013-06-03,NaN,Garden Fresh - Large,16.0,Salad
6,101964,2013-06-03,NaN,The Works - Small,9.0,Pizza
7,101964,2013-06-03,NaN,Buffalo Chicken - Small,9.0,Pizza
8,101964,2013-06-03,NaN,Soda - Small,2.0,Drink
9,101964,2013-06-03,NaN,Soda - Large,3.0,Drink


In [123]:
df_group = df_merge.groupby(by=['TicketID', 'Date'])['Type'].apply(lambda x: ','.join(x)).reset_index()
df_group

,TicketID,Date,Type
0,101964,2013-06-03,"Side,Pizza,Salad,Salad,Salad,Salad,Pizza,Pizza..."
1,101966,2013-06-03,"Side,Pizza,Side,Side,Salad,Salad,Burger,Pizza,..."
2,101969,2013-06-03,"Soup,Soup,Pizza,Pizza,Drink,Drink"
3,101979,2013-06-03,"Pizza,Side,Soup,Soup,Salad,Burger,Burger,Pizza..."
4,101982,2013-06-03,"Burger,Pizza,Pizza,Drink,Drink"
...,...,...,...
17972,331453,2013-12-28,"Salad,Salad,Salad,Pizza,Pizza,Pizza,Drink"
17973,331454,2013-12-28,"Side,Side,Salad,Pizza,Pizza,Pizza,Pizza,Drink"
17974,331459,2013-12-28,"Pizza,Burger,Pizza,Pizza,Drink,Drink,Drink"
17975,331461,2013-12-28,"Side,Side,Salad,Pizza,Pizza,Pizza,Drink"


In [124]:
df_deal = df_group[(df_group['Type'].str.contains('Burger|Pizza', regex=True)) & (df_group['Type'].str.contains('Side')) & (df_group['Type'].str.contains('Drink'))]
df_deal

,TicketID,Date,Type
0,101964,2013-06-03,"Side,Pizza,Salad,Salad,Salad,Salad,Pizza,Pizza..."
1,101966,2013-06-03,"Side,Pizza,Side,Side,Salad,Salad,Burger,Pizza,..."
3,101979,2013-06-03,"Pizza,Side,Soup,Soup,Salad,Burger,Burger,Pizza..."
6,101993,2013-06-03,"Side,Side,Soup,Salad,Salad,Burger,Pizza,Pizza,..."
7,101995,2013-06-03,"Side,Pizza,Side,Soup,Salad,Salad,Salad,Pizza,P..."
...,...,...,...
17969,331446,2013-12-28,"Side,Side,Side,Side,Side,Pizza,Pizza,Pizza,Piz..."
17970,331447,2013-12-28,"Side,Burger,Pizza,Pizza,Pizza,Drink,Drink"
17973,331454,2013-12-28,"Side,Side,Salad,Pizza,Pizza,Pizza,Pizza,Drink"
17975,331461,2013-12-28,"Side,Side,Salad,Pizza,Pizza,Pizza,Drink"


In [125]:
df.groupby(by='TicketID').sum('Price')
df

,TicketID,Date,MemberID,Desc,Price
5528,101964,2013-06-03,NaN,Mozzarella Sticks,7.0
5529,101964,2013-06-03,NaN,Stromboli,7.5
5530,101964,2013-06-03,NaN,House Salad - Half,3.0
5531,101964,2013-06-03,NaN,House Salad - Whole,6.0
5532,101964,2013-06-03,NaN,Garden Fresh - Medium,14.0
...,...,...,...,...,...
269370,331462,2013-12-28,NaN,The Works - Medium,14.0
269371,331462,2013-12-28,NaN,Soda - Small,2.0
269372,331462,2013-12-28,NaN,Soda - Small,2.0
269373,331462,2013-12-28,NaN,Soda - Large,3.0


In [126]:
results = pd.DataFrame({'Potential Meal Deal': [df_deal.count()['TicketID']], 'Total': [df.count()['TicketID']]})
results['%'] = results['Potential Meal Deal'] / results['Total']
results

,Potential Meal Deal,Total,%
0,12751,157216,0.081105
